# Task 1 Exact Pattern Classification

This notebook predicts the stimulation pattern ID as a nominal 16-class label. Pattern IDs are random identifiers and are **not** treated as adjacent points on a class circle.

Final training is configured for the shared final dataset `N5_DIV40.h5` with `group_data=False` and `test_mode=False`. The corresponding `_test` file is loaded only as a separate inference/submission reference because the final-task files intentionally omit one side of the supervised pairs.

The excitatory-loop prior is used only where it is physically meaningful: electrode coordinates, circular electrode features, and kNN/clockwise/counterclockwise/self graph relations over the network topology.

The model remains multi-network-capable, but this final configuration intentionally trains only on Network 5 DIV 40. Training uses validation macro-F1 as the primary metric, matching the final Task 1 F1 scoring emphasis; exact accuracy remains a secondary diagnostic.


In [ ]:
import sys
from pathlib import Path

# Make local Final_Task utilities importable even when Jupyter starts outside this folder.
final_task_dir = Path.cwd()
if not (final_task_dir / "utils").exists():
    candidates = [
        Path.home() / "pands_ibnnwml" / "Final_Task",
        Path.cwd() / "pands_ibnnwml" / "Final_Task",
        Path.cwd().parent / "Final_Task",
    ]
    final_task_dir = next((path for path in candidates if (path / "utils").exists()), final_task_dir)

if str(final_task_dir) not in sys.path:
    sys.path.insert(0, str(final_task_dir))

from utils.data import load_data, save_data
from utils.plotting import create_post_stim_raster_plot, map_colour_to_electrode


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
import getpass
import json
import pandas as pd
from datetime import datetime

In [ ]:
# Final shared-data setup for supervised Task 1 training.
NETWORK_CANDIDATES = [5]
DIV = 40
group_data = False
TRAIN_TEST_MODE = False  # supervised training file: shared/N5_DIV40.h5
LOAD_FINAL_TEST_DATA = True
FINAL_TEST_MODE = True   # inference/submission file: shared/N5_DIV40_test.h5

# Split options:
# - "random": train/validation split stratified by network/frequency/pattern.
# - "leave_network_out": supported only when NETWORK_CANDIDATES has >=2 networks.
SPLIT_MODE = "random"
HOLDOUT_NETWORK = None
VAL_FRAC = 0.10
RANDOM_SEED = 42

print("Task 1 final shared-data setup")
print("Training networks:", NETWORK_CANDIDATES, "DIV", DIV, "group_data", group_data, "test_mode", TRAIN_TEST_MODE)
print("Split mode:", SPLIT_MODE, "holdout:", HOLDOUT_NETWORK if SPLIT_MODE == "leave_network_out" else None)
print("Final _test file loaded for inference only:", LOAD_FINAL_TEST_DATA)


In [ ]:
def array_shape_or_none(x):
    return None if x is None else tuple(np.asarray(x).shape)

network_records = []
for network in NETWORK_CANDIDATES:
    try:
        data = load_data(network, DIV, group_data, test_mode=TRAIN_TEST_MODE)
        stimulation_parameters, stimulation_patterns, responses, stimulation_times, impedance_map, electrodes = data
        if stimulation_parameters is None or responses is None:
            raise ValueError(
                "Supervised Task 1 training requires stimulation parameters and responses. "
                "Use test_mode=False for the final shared training file."
            )
        responses = np.asarray(responses)
        if responses.ndim == 4 and responses.shape[1] == 1:
            responses = responses[:, 0]
        if responses.ndim != 3:
            raise ValueError(f"Expected [trials,electrodes,time], got {responses.shape}")
        record = dict(
            network=int(network),
            stimulation_parameters=np.asarray(stimulation_parameters, dtype=np.float32),
            stimulation_patterns=np.asarray(stimulation_patterns, dtype=np.int64),
            responses=responses.astype(np.float32),
            stimulation_times=np.asarray(stimulation_times) if stimulation_times is not None else None,
            impedance_map=impedance_map,
            electrodes=np.asarray(electrodes),
        )
        network_records.append(record)
        print(
            f"Loaded final training network {network}: responses={responses.shape}, "
            f"patterns={np.unique(record['stimulation_patterns']).astype(int).tolist()}, "
            f"freqs={len(np.unique(record['stimulation_parameters'][:, 0]))}"
        )
    except Exception as e:
        print(f"Skipping network {network}: {repr(e)}")

if len(network_records) < 1:
    raise RuntimeError("Need the shared final Task 1 training data N5_DIV40.h5 to train.")
if SPLIT_MODE == "leave_network_out" and len(network_records) < 2:
    raise RuntimeError("leave_network_out requires at least two training networks; use SPLIT_MODE='random' for final N5_DIV40 training.")

available_networks = [r["network"] for r in network_records]
network_to_local = {net: i for i, net in enumerate(available_networks)}
print("Available training networks:", available_networks)
print("Network id mapping:", network_to_local)

final_task1_test_data = None
if LOAD_FINAL_TEST_DATA:
    try:
        final_task1_test_data = load_data(NETWORK_CANDIDATES[0], DIV, group_data, test_mode=FINAL_TEST_MODE)
        test_stimulation_parameters, test_stimulation_patterns, test_responses, test_stimulation_times, test_impedance_map, test_electrodes = final_task1_test_data
        print("Loaded final Task 1 _test file for inference/submission reference only:")
        print("  stimulation_parameters:", array_shape_or_none(test_stimulation_parameters))
        print("  stimulation_patterns:", array_shape_or_none(test_stimulation_patterns))
        print("  responses:", array_shape_or_none(test_responses))
        print("  stimulation_times:", array_shape_or_none(test_stimulation_times))
    except Exception as e:
        print("WARNING: could not load final Task 1 _test file:", repr(e))


In [ ]:
# 2. Model Definitions and Data Preparation

In [ ]:
class MultiRelationGraphBlock(nn.Module):
    def __init__(self, hidden_dim, n_relations, dropout=0.20):
        super().__init__()
        self.n_relations = int(n_relations)
        self.update = nn.Sequential(
            nn.Linear(hidden_dim * (1 + self.n_relations), hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, h, adj, mask):
        # adj: [batch, relation, electrode, electrode]
        if adj.ndim == 3:
            adj = adj.unsqueeze(1)
        neigh = torch.einsum("brij,bjd->brid", adj, h)
        neigh = neigh.permute(0, 2, 1, 3).flatten(start_dim=2)
        delta = self.update(torch.cat([h, neigh], dim=-1))
        h = self.norm(h + delta)
        return h * mask


class MultiNetworkTask1Classifier(nn.Module):
    """Circular-topology final-data graph-temporal classifier for Task 1.

    The PDF describes recurrent excitatory loop motifs in the network topology and
    response propagation. Pattern IDs are nominal class labels, so topology is used
    only for electrode/response encoding, not for class adjacency. Multi-network
    training shares the response encoder while adding network-specific residual
    classifier heads because label semantics and electrode geometry are domain
    dependent.
    """

    def __init__(
        self,
        n_outputs,
        n_networks,
        n_max_electrodes,
        n_time,
        coords_by_network,
        electrode_circular_by_network,
        adjacency_by_network,
        electrode_static_by_network,
        electrode_mask_by_network,
        class_prototypes=None,
        n_base_channels=48,
        hidden_dim=128,
        electrode_emb_dim=20,
        network_emb_dim=16,
        dropout=0.36,
    ):
        super().__init__()
        self.n_outputs = int(n_outputs)
        self.n_networks = int(n_networks)
        self.n_max_electrodes = int(n_max_electrodes)
        self.n_time = int(n_time)
        self.hidden_dim = int(hidden_dim)

        self.register_buffer("coords_by_network", torch.as_tensor(coords_by_network, dtype=torch.float32))
        self.register_buffer("electrode_circular_by_network", torch.as_tensor(electrode_circular_by_network, dtype=torch.float32))
        self.register_buffer("adjacency_by_network", torch.as_tensor(adjacency_by_network, dtype=torch.float32))
        self.register_buffer("electrode_static_by_network", torch.as_tensor(electrode_static_by_network, dtype=torch.float32))
        self.register_buffer("electrode_mask_by_network", torch.as_tensor(electrode_mask_by_network, dtype=torch.float32))

        if class_prototypes is None:
            class_prototypes = np.zeros((n_networks, n_outputs, n_max_electrodes * n_time), dtype=np.float32)
        class_prototypes = torch.as_tensor(class_prototypes, dtype=torch.float32)
        class_prototypes = F.normalize(class_prototypes, dim=-1)
        self.register_buffer("class_prototypes", class_prototypes)
        self.prototype_scale = nn.Parameter(torch.tensor(4.0))

        self.temporal_layers = nn.Sequential(
            nn.Conv2d(1, n_base_channels, kernel_size=(1, 9), padding=(0, 4), bias=False),
            nn.BatchNorm2d(n_base_channels),
            nn.SiLU(),
            nn.Dropout2d(0.04),
            nn.Conv2d(n_base_channels, n_base_channels, kernel_size=(1, 7), padding=(0, 3), bias=False),
            nn.BatchNorm2d(n_base_channels),
            nn.SiLU(),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Conv2d(n_base_channels, n_base_channels * 2, kernel_size=(1, 5), padding=(0, 2), bias=False),
            nn.BatchNorm2d(n_base_channels * 2),
            nn.SiLU(),
            nn.Dropout2d(0.08),
            nn.Conv2d(n_base_channels * 2, n_base_channels * 2, kernel_size=(1, 5), padding=(0, 2), bias=False),
            nn.BatchNorm2d(n_base_channels * 2),
            nn.SiLU(),
        )

        self.electrode_emb = nn.Embedding(n_max_electrodes, electrode_emb_dim)
        self.network_emb = nn.Embedding(n_networks, network_emb_dim)
        temporal_feature_dim = n_base_channels * 2 * 2
        # temporal + xy coords + circular electrode features + static electrode features + ids
        electrode_feature_dim = temporal_feature_dim + 2 + 3 + 3 + electrode_emb_dim + network_emb_dim
        self.electrode_mlp = nn.Sequential(
            nn.Linear(electrode_feature_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.25),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        n_relations = int(self.adjacency_by_network.shape[1]) if self.adjacency_by_network.ndim == 4 else 1
        self.graph_blocks = nn.ModuleList([
            MultiRelationGraphBlock(hidden_dim, n_relations=n_relations, dropout=dropout * 0.25)
            for _ in range(2)
        ])

        freq_dim = 7
        pooled_dim = hidden_dim * 3 + freq_dim + network_emb_dim
        self.classifier_features = nn.Sequential(
            nn.Linear(pooled_dim, 384),
            nn.BatchNorm1d(384),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(384, 192),
            nn.BatchNorm1d(192),
            nn.GELU(),
            nn.Dropout(dropout * 0.55),
        )
        self.shared_class_head = nn.Linear(192, n_outputs)
        self.network_class_weight = nn.Parameter(torch.zeros(n_networks, n_outputs, 192))
        self.network_class_bias = nn.Parameter(torch.zeros(n_networks, n_outputs))
        nn.init.normal_(self.network_class_weight, mean=0.0, std=1e-3)

    def frequency_features(self, freq):
        freq = freq.view(-1, 1).to(dtype=self.coords_by_network.dtype)
        return torch.cat([
            freq,
            freq.square(),
            torch.sin(np.pi * freq),
            torch.cos(np.pi * freq),
            torch.sin(2 * np.pi * freq),
            torch.cos(2 * np.pi * freq),
            torch.sin(4 * np.pi * freq),
        ], dim=1)

    def forward(self, x, freq=None, network_id=None):
        b = x.shape[0]
        if network_id is None:
            network_id = torch.zeros(b, device=x.device, dtype=torch.long)
        network_id = network_id.to(device=x.device, dtype=torch.long)
        raw_x = x

        x = self.temporal_layers(x)
        avg_t = x.mean(dim=-1).transpose(1, 2)
        max_t = x.amax(dim=-1).transpose(1, 2)
        temporal = torch.cat([avg_t, max_t], dim=-1)

        ids = torch.arange(self.n_max_electrodes, device=x.device)
        coords = self.coords_by_network[network_id].to(x.device)
        circular = self.electrode_circular_by_network[network_id].to(x.device)
        static = self.electrode_static_by_network[network_id].to(x.device)
        mask = self.electrode_mask_by_network[network_id].to(x.device).unsqueeze(-1)
        adj = self.adjacency_by_network[network_id].to(x.device)
        electrode_emb = self.electrode_emb(ids)[None, :, :].expand(b, -1, -1)
        network_emb = self.network_emb(network_id)
        network_e = network_emb[:, None, :].expand(-1, self.n_max_electrodes, -1)

        h = self.electrode_mlp(torch.cat([temporal, coords, circular, static, electrode_emb, network_e], dim=-1))
        h = h * mask
        for block in self.graph_blocks:
            h = block(h, adj, mask)

        denom = mask.sum(dim=1).clamp_min(1.0)
        mean_pool = h.sum(dim=1) / denom
        centered = (h - mean_pool[:, None, :]) * mask
        std_pool = torch.sqrt((centered.square().sum(dim=1) / denom).clamp_min(1e-8))
        max_pool = h.masked_fill(mask <= 0, -1e9).amax(dim=1)

        if freq is None:
            freq = torch.zeros(b, device=x.device, dtype=x.dtype)
        else:
            freq = freq.to(device=x.device, dtype=x.dtype)
        freq_feat = self.frequency_features(freq)

        features = torch.cat([mean_pool, max_pool, std_pool, freq_feat, network_emb], dim=1)
        cls_features = self.classifier_features(features)
        class_logits = self.shared_class_head(cls_features)
        local_w = self.network_class_weight[network_id]
        local_b = self.network_class_bias[network_id]
        class_logits = class_logits + torch.bmm(local_w, cls_features.unsqueeze(-1)).squeeze(-1) + local_b

        if self.class_prototypes.numel() > 0:
            raw_flat = F.normalize(raw_x.flatten(start_dim=1), dim=1)
            protos = self.class_prototypes[network_id].to(raw_x.device)
            proto_logits = torch.bmm(protos, raw_flat.unsqueeze(-1)).squeeze(-1)
            class_logits = class_logits + self.prototype_scale * proto_logits

        return class_logits



In [ ]:
# Hyperparameters for exact-pattern final-data training with electrode-loop topology.
n_epochs = 140
batch_size = 256
learning_rate = 6e-4
weight_decay = 3e-4
n_classes = 16

# Pattern IDs are nominal. Do not smooth labels around an inferred pattern circle.
CLASS_LABEL_SMOOTHING = 0.0

# Stop when validation macro-F1 has plateaued. The checkpoint tracks the
# numerically best macro-F1 seen in the run; accuracy is logged as a secondary metric.
EARLY_STOP_PATIENCE = 14
EARLY_STOP_MIN_DELTA = 1e-4
SCHEDULER_PATIENCE = 6

# Regularization/augmentation: keep light response-domain regularization, since
# previous Task 1 runs overfit easily; keep final-data regularization light but present.
RESPONSE_NOISE_STD = 0.02
ELECTRODE_DROPOUT_PROB = 0.025
TIME_DROPOUT_PROB = 0.02

print("Final-data exact Task 1 hyperparameters:")
print(dict(
    n_epochs=n_epochs,
    batch_size=batch_size,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    class_label_smoothing=CLASS_LABEL_SMOOTHING,
    early_stop_patience=EARLY_STOP_PATIENCE,
    early_stop_min_delta=EARLY_STOP_MIN_DELTA,  # F1 points, not accuracy percent
    primary_metric="macro_f1",
    scheduler_patience=SCHEDULER_PATIENCE,
    response_noise_std=RESPONSE_NOISE_STD,
    electrode_dropout_prob=ELECTRODE_DROPOUT_PROB,
    time_dropout_prob=TIME_DROPOUT_PROB,
))


In [ ]:
from torch.utils.data import Dataset, DataLoader

val_frac = VAL_FRAC
rng = np.random.default_rng(RANDOM_SEED)

n_networks = len(network_records)
n_max_electrodes = max(np.asarray(r["responses"]).shape[1] for r in network_records)
n_time = int(np.asarray(network_records[0]["responses"]).shape[2])
print(f"Padding all networks to max electrodes={n_max_electrodes}, time bins={n_time}")


def extract_electrode_xy(electrodes, impedance_map, n_electrodes):
    electrodes_arr = np.asarray(electrodes)
    if electrodes_arr.ndim == 2 and electrodes_arr.shape[0] == n_electrodes and electrodes_arr.shape[1] >= 2:
        return electrodes_arr[:, :2].astype(np.float32)
    imp = np.squeeze(np.asarray(impedance_map))
    if imp.ndim != 2:
        return np.stack([np.arange(n_electrodes), np.zeros(n_electrodes)], axis=1).astype(np.float32)
    width = imp.shape[1]
    ids = electrodes_arr.reshape(-1)[:n_electrodes].astype(int)
    return np.stack([ids % width, ids // width], axis=1).astype(np.float32)


def build_knn_adjacency(coords_norm, k=8, self_weight=1.0):
    coords_arr = np.asarray(coords_norm, dtype=np.float32)
    diff = coords_arr[:, None, :] - coords_arr[None, :, :]
    dist = np.sqrt(np.sum(diff * diff, axis=-1))
    np.fill_diagonal(dist, np.inf)
    nn_idx = np.argsort(dist, axis=1)[:, :min(k, max(1, coords_arr.shape[0] - 1))]
    adj = np.zeros((coords_arr.shape[0], coords_arr.shape[0]), dtype=np.float32)
    rows = np.arange(coords_arr.shape[0])[:, None]
    adj[rows, nn_idx] = 1.0
    adj = np.maximum(adj, adj.T)
    np.fill_diagonal(adj, self_weight)
    adj = adj / (adj.sum(axis=1, keepdims=True) + 1e-6)
    return adj.astype(np.float32)


def build_circular_electrode_features(coords_norm):
    coords_arr = np.asarray(coords_norm, dtype=np.float32)
    center = coords_arr.mean(axis=0, keepdims=True)
    shifted = coords_arr - center
    angles = np.arctan2(shifted[:, 1], shifted[:, 0]).astype(np.float32)
    radius = np.sqrt(np.sum(shifted * shifted, axis=1)).astype(np.float32)
    radius = ((radius - radius.mean()) / (radius.std() + 1e-6)).astype(np.float32)
    features = np.stack([np.sin(angles), np.cos(angles), radius], axis=1).astype(np.float32)
    return angles, features


def _row_normalize(adj):
    return (adj / (adj.sum(axis=1, keepdims=True) + 1e-6)).astype(np.float32)


def build_ring_relation_adjacency(coords_norm, k=8):
    n = int(coords_norm.shape[0])
    knn = build_knn_adjacency(coords_norm, k=k, self_weight=1.0)
    angles, _ = build_circular_electrode_features(coords_norm)
    order = np.argsort(angles)
    clockwise = np.zeros((n, n), dtype=np.float32)
    counterclockwise = np.zeros((n, n), dtype=np.float32)
    for rank, electrode_idx in enumerate(order):
        next_idx = order[(rank + 1) % n]
        prev_idx = order[(rank - 1) % n]
        clockwise[electrode_idx, next_idx] = 1.0
        counterclockwise[electrode_idx, prev_idx] = 1.0
    self_adj = np.eye(n, dtype=np.float32)
    return np.stack([knn, _row_normalize(clockwise), _row_normalize(counterclockwise), self_adj], axis=0).astype(np.float32)


X_pieces = []
y_pieces = []
z_pieces = []
network_id_pieces = []
electrode_mask_by_network = np.zeros((n_networks, n_max_electrodes), dtype=np.float32)
coords_by_network = np.zeros((n_networks, n_max_electrodes, 2), dtype=np.float32)
electrode_circular_by_network = np.zeros((n_networks, n_max_electrodes, 3), dtype=np.float32)
adjacency_by_network = np.zeros((n_networks, 4, n_max_electrodes, n_max_electrodes), dtype=np.float32)
electrode_impedance_by_network = np.zeros((n_networks, n_max_electrodes), dtype=np.float32)

for local_id, record in enumerate(network_records):
    X = np.asarray(record["responses"], dtype=np.float32)
    X = np.log1p(X)
    m, n_e, t = X.shape
    if t != n_time:
        raise RuntimeError(f"Network {record['network']} has {t} time bins, expected {n_time}")
    padded = np.zeros((m, 1, n_max_electrodes, n_time), dtype=np.float32)
    padded[:, 0, :n_e, :] = X
    X_pieces.append(padded)
    y_pieces.append(record["stimulation_patterns"].astype(np.int64))
    z_pieces.append(record["stimulation_parameters"].astype(np.float32))
    network_id_pieces.append(np.full(m, local_id, dtype=np.int64))
    electrode_mask_by_network[local_id, :n_e] = 1.0

    coords = extract_electrode_xy(record["electrodes"], record["impedance_map"], n_e)
    coord_mean = coords.mean(axis=0, keepdims=True)
    coord_std = coords.std(axis=0, keepdims=True) + 1e-6
    coords_norm = ((coords - coord_mean) / coord_std).astype(np.float32)
    _, circular_features = build_circular_electrode_features(coords_norm)
    coords_by_network[local_id, :n_e, :] = coords_norm
    electrode_circular_by_network[local_id, :n_e, :] = circular_features
    adjacency_by_network[local_id, :, :n_e, :n_e] = build_ring_relation_adjacency(coords_norm, k=8)

    imp = np.squeeze(np.asarray(record["impedance_map"]))
    if imp.ndim == 2:
        xs = np.clip(coords[:, 0].round().astype(int), 0, imp.shape[1] - 1)
        ys = np.clip(coords[:, 1].round().astype(int), 0, imp.shape[0] - 1)
        eimp = imp[ys, xs].astype(np.float32)
    else:
        eimp = np.zeros(n_e, dtype=np.float32)
    eimp = np.nan_to_num(eimp, nan=float(np.nanmean(eimp)))
    eimp = (eimp - eimp.mean()) / (eimp.std() + 1e-6)
    electrode_impedance_by_network[local_id, :n_e] = eimp

X_all = np.concatenate(X_pieces, axis=0)
y_all = np.concatenate(y_pieces, axis=0).astype(np.int64)
z_all = np.concatenate(z_pieces, axis=0).astype(np.float32)
network_id_all = np.concatenate(network_id_pieces, axis=0).astype(np.int64)
print("Combined samples:", X_all.shape[0], "X:", X_all.shape, "networks:", available_networks)

all_indices = np.arange(len(y_all))
if SPLIT_MODE == "random":
    strata = np.array([f"{network_id_all[i]}_{z_all[i,0]:.4g}_{y_all[i]}" for i in all_indices])
    val_indices = []
    train_indices = []
    for stratum in np.unique(strata):
        idx = np.flatnonzero(strata == stratum)
        rng.shuffle(idx)
        n_val = max(1, int(round(len(idx) * val_frac)))
        val_indices.append(idx[:n_val])
        train_indices.append(idx[n_val:])
    val_idx_np = np.concatenate(val_indices)
    trn_idx_np = np.concatenate(train_indices)
elif SPLIT_MODE == "leave_network_out":
    if HOLDOUT_NETWORK not in network_to_local:
        raise RuntimeError(f"HOLDOUT_NETWORK={HOLDOUT_NETWORK} is not available. Available={available_networks}")
    holdout_local = network_to_local[HOLDOUT_NETWORK]
    val_idx_np = np.flatnonzero(network_id_all == holdout_local)
    trn_idx_np = np.flatnonzero(network_id_all != holdout_local)
else:
    raise ValueError(f"Unknown SPLIT_MODE: {SPLIT_MODE}")

rng.shuffle(val_idx_np)
rng.shuffle(trn_idx_np)
print("Train samples:", len(trn_idx_np), "Val samples:", len(val_idx_np))
print("Train networks:", sorted({available_networks[i] for i in np.unique(network_id_all[trn_idx_np])}))
print("Val networks:", sorted({available_networks[i] for i in np.unique(network_id_all[val_idx_np])}))

train_masks = electrode_mask_by_network[network_id_all[trn_idx_np]][:, None, :, None]
valid_count = float(train_masks.sum() * n_time)
train_values = X_all[trn_idx_np]
mean = float((train_values * train_masks).sum() / max(valid_count, 1.0))
var = float((((train_values - mean) ** 2) * train_masks).sum() / max(valid_count, 1.0))
std = float(np.sqrt(var) + 1e-6)
print(f"Response normalization: mean={mean:.5f}, std={std:.5f}")
X_all = ((X_all - mean) / std).astype(np.float32)
X_all *= electrode_mask_by_network[network_id_all][:, None, :, None]

freq_mean = float(z_all[trn_idx_np, 0].mean())
freq_std = float(z_all[trn_idx_np, 0].std() + 1e-6)
print(f"Frequency normalization: mean={freq_mean:.3f}, std={freq_std:.3f}")

# Train-only per-network electrode rate after response normalization.
electrode_rate_by_network = np.zeros((n_networks, n_max_electrodes), dtype=np.float32)
for local_id in range(n_networks):
    n_e = int(electrode_mask_by_network[local_id].sum())
    mask_idx = trn_idx_np[network_id_all[trn_idx_np] == local_id]
    if len(mask_idx):
        rate = X_all[mask_idx, 0, :n_e, :].mean(axis=(0, 2)).astype(np.float32)
    else:
        rate = np.zeros(n_e, dtype=np.float32)
    rate = (rate - rate.mean()) / (rate.std() + 1e-6)
    electrode_rate_by_network[local_id, :n_e] = rate

# Use angular electrode order as the static index, because the physical motif is a loop.
electrode_ring_index_by_network = np.zeros((n_networks, n_max_electrodes), dtype=np.float32)
for local_id in range(n_networks):
    n_e = int(electrode_mask_by_network[local_id].sum())
    angles = np.arctan2(electrode_circular_by_network[local_id, :n_e, 0], electrode_circular_by_network[local_id, :n_e, 1])
    order = np.argsort(angles)
    ring_rank = np.empty(n_e, dtype=np.float32)
    ring_rank[order] = np.linspace(-1.0, 1.0, n_e, dtype=np.float32)
    electrode_ring_index_by_network[local_id, :n_e] = ring_rank

electrode_static_by_network = np.stack([electrode_rate_by_network, electrode_impedance_by_network, electrode_ring_index_by_network], axis=-1).astype(np.float32)

class_prototypes = np.zeros((n_networks, n_classes, n_max_electrodes * n_time), dtype=np.float32)
global_class_prototypes = np.zeros((n_classes, n_max_electrodes * n_time), dtype=np.float32)
for class_id in range(n_classes):
    mask_idx = trn_idx_np[y_all[trn_idx_np] == class_id]
    if len(mask_idx):
        proto = X_all[mask_idx, 0].mean(axis=0).reshape(-1)
        global_class_prototypes[class_id] = proto.astype(np.float32)

class_counts_by_network = np.zeros((n_networks, n_classes), dtype=np.int64)
for local_id in range(n_networks):
    for class_id in range(n_classes):
        mask_idx = trn_idx_np[(network_id_all[trn_idx_np] == local_id) & (y_all[trn_idx_np] == class_id)]
        class_counts_by_network[local_id, class_id] = int(len(mask_idx))
        if len(mask_idx):
            proto = X_all[mask_idx, 0].mean(axis=0).reshape(-1)
            norm = np.linalg.norm(proto) + 1e-8
            class_prototypes[local_id, class_id] = (proto / norm).astype(np.float32)
        else:
            proto = global_class_prototypes[class_id]
            norm = np.linalg.norm(proto) + 1e-8
            class_prototypes[local_id, class_id] = (proto / norm).astype(np.float32)

print("Pattern labels are treated as nominal IDs; no pattern-circle smoothing or angle target is used.")
print("Train class counts by network:")
print(pd.DataFrame(class_counts_by_network, index=available_networks, columns=[f"pattern_{i}" for i in range(n_classes)]).to_string())

class MultiNetworkTask1Dataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, j):
        i = self.indices[j]
        return (
            torch.from_numpy(X_all[i]),
            torch.tensor(y_all[i], dtype=torch.long),
            torch.from_numpy(z_all[i]),
            torch.tensor(network_id_all[i], dtype=torch.long),
        )

train_dataset = MultiNetworkTask1Dataset(trn_idx_np)
val_dataset = MultiNetworkTask1Dataset(val_idx_np)
val_true_labels = y_all[val_idx_np]
val_frequencies = z_all[val_idx_np, 0].astype(np.float32)
val_network_ids = network_id_all[val_idx_np].astype(np.int64)
val_network_labels = np.array([available_networks[i] for i in val_network_ids], dtype=np.int64)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

model = MultiNetworkTask1Classifier(
    n_outputs=n_classes,
    n_networks=n_networks,
    n_max_electrodes=n_max_electrodes,
    n_time=n_time,
    coords_by_network=coords_by_network,
    electrode_circular_by_network=electrode_circular_by_network,
    adjacency_by_network=adjacency_by_network,
    electrode_static_by_network=electrode_static_by_network,
    electrode_mask_by_network=electrode_mask_by_network,
    class_prototypes=class_prototypes,
    n_base_channels=48,
    hidden_dim=128,
    dropout=0.36,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=SCHEDULER_PATIENCE)
print(f"The exact-label loop-topology final-data model has {sum(p.numel() for p in model.parameters()):,} parameters.")



In [ ]:
# 3. Training and Validation Loop

In [ ]:
import os
import getpass
import json
import sys
import subprocess
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm


class TeeStream:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()
        return len(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return any(getattr(stream, "isatty", lambda: False)() for stream in self.streams)


def task1_classification_metrics(y_true, y_pred, n_classes):
    """Return multiclass F1 metrics using one-vs-rest TP/FP/FN per pattern.

    The final-data PDF defines F1 from TP/FP/FN. For 16-way single-label
    classification we use macro-F1 as the primary metric; micro-F1 is also
    reported and equals accuracy in this setting.
    """
    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.int64).reshape(-1)
    labels = np.arange(int(n_classes), dtype=np.int64)
    rows = []
    for label in labels:
        true_pos = y_true == label
        pred_pos = y_pred == label
        tp = int(np.logical_and(true_pos, pred_pos).sum())
        fp = int(np.logical_and(~true_pos, pred_pos).sum())
        fn = int(np.logical_and(true_pos, ~pred_pos).sum())
        support = int(true_pos.sum())
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        f1 = 0.0 if precision + recall == 0 else 2.0 * precision * recall / (precision + recall)
        rows.append(dict(
            pattern=int(label),
            n_samples=support,
            tp=tp,
            fp=fp,
            fn=fn,
            precision=float(precision),
            recall=float(recall),
            f1=float(f1),
            precision_percent=100.0 * float(precision),
            recall_percent=100.0 * float(recall),
            f1_percent=100.0 * float(f1),
        ))
    per_class = pd.DataFrame(rows)
    support = per_class["n_samples"].to_numpy(dtype=np.float64)
    f1_values = per_class["f1"].to_numpy(dtype=np.float64)
    tp_total = int(per_class["tp"].sum())
    fp_total = int(per_class["fp"].sum())
    fn_total = int(per_class["fn"].sum())
    micro_precision = tp_total / max(1, tp_total + fp_total)
    micro_recall = tp_total / max(1, tp_total + fn_total)
    micro_f1 = 0.0 if micro_precision + micro_recall == 0 else 2.0 * micro_precision * micro_recall / (micro_precision + micro_recall)
    macro_f1 = float(f1_values.mean()) if len(f1_values) else 0.0
    weighted_f1 = float((f1_values * support).sum() / max(1.0, support.sum()))
    accuracy = float((y_true == y_pred).mean()) if len(y_true) else 0.0
    metrics = dict(
        accuracy=accuracy,
        macro_f1=macro_f1,
        weighted_f1=weighted_f1,
        micro_f1=float(micro_f1),
        macro_f1_percent=100.0 * macro_f1,
        weighted_f1_percent=100.0 * weighted_f1,
        micro_f1_percent=100.0 * float(micro_f1),
        accuracy_percent=100.0 * accuracy,
    )
    return metrics, per_class


def write_task1_artifacts(output_dir, history_rows, summary, best_predictions=None, best_targets=None, best_frequencies=None, best_networks=None, best_network_local_ids=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(output_dir / "training_history.csv", index=False)
    with open(output_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    pd.DataFrame(class_counts_by_network, index=available_networks, columns=[f"pattern_{i}" for i in range(n_classes)]).reset_index(names="network").to_csv(output_dir / "train_class_counts_by_network.csv", index=False)
    if best_predictions is None or best_targets is None:
        return
    pred_df = pd.DataFrame({
        "sample_index": np.arange(len(best_targets)),
        "network": best_networks if best_networks is not None else np.nan,
        "frequency": best_frequencies if best_frequencies is not None else np.nan,
        "true_pattern": best_targets.astype(int),
        "pred_pattern": best_predictions.astype(int),
        "correct": (best_predictions == best_targets).astype(int),
    })
    pred_df.to_csv(output_dir / "best_validation_predictions.csv", index=False)
    best_metric_summary, best_per_class_f1 = task1_classification_metrics(best_targets, best_predictions, n_classes)
    pd.DataFrame([best_metric_summary]).to_csv(output_dir / "f1_summary.csv", index=False)
    best_per_class_f1.to_csv(output_dir / "f1_by_pattern.csv", index=False)
    rows = []
    for pattern in np.unique(best_targets):
        mask = best_targets == pattern
        acc = float((best_predictions[mask] == best_targets[mask]).mean())
        rows.append({"pattern": int(pattern), "n_samples": int(mask.sum()), "accuracy": acc, "accuracy_percent": 100.0 * acc})
    pd.DataFrame(rows).to_csv(output_dir / "accuracy_by_pattern.csv", index=False)
    if best_networks is not None:
        rows = []
        for network in np.unique(best_networks):
            mask = best_networks == network
            acc = float((best_predictions[mask] == best_targets[mask]).mean())
            rows.append({"network": int(network), "n_samples": int(mask.sum()), "accuracy": acc, "accuracy_percent": 100.0 * acc})
        pd.DataFrame(rows).to_csv(output_dir / "accuracy_by_network.csv", index=False)
        rows = []
        for network in np.unique(best_networks):
            for pattern in np.unique(best_targets):
                mask = (best_networks == network) & (best_targets == pattern)
                if mask.any():
                    acc = float((best_predictions[mask] == best_targets[mask]).mean())
                    rows.append({"network": int(network), "pattern": int(pattern), "n_samples": int(mask.sum()), "accuracy": acc, "accuracy_percent": 100.0 * acc})
        pd.DataFrame(rows).to_csv(output_dir / "accuracy_by_network_pattern.csv", index=False)
    rows = []
    if best_frequencies is not None:
        for freq in np.unique(best_frequencies):
            mask = best_frequencies == freq
            acc = float((best_predictions[mask] == best_targets[mask]).mean())
            rows.append({"frequency": float(freq), "n_samples": int(mask.sum()), "accuracy": acc, "accuracy_percent": 100.0 * acc})
    pd.DataFrame(rows).to_csv(output_dir / "accuracy_by_frequency.csv", index=False)
    cm = pd.crosstab(pd.Series(best_targets.astype(int), name="true_pattern"), pd.Series(best_predictions.astype(int), name="pred_pattern"), dropna=False)
    cm.to_csv(output_dir / "confusion_matrix.csv")
    np.savez_compressed(
        output_dir / "best_validation_arrays.npz",
        predictions=best_predictions,
        targets=best_targets,
        frequencies=best_frequencies,
        networks=best_networks,
        network_local_ids=best_network_local_ids,
    )


def get_idle_cuda_devices(max_devices=2, max_util_percent=1, min_free_memory_mb=6500, util_samples=5, sample_interval_sec=1.0, fallback_allow_any_visible_gpu_if_nvidia_smi_fails=False):
    if not torch.cuda.is_available():
        return []
    n_torch = torch.cuda.device_count()
    visible_physical_ids = _parse_visible_cuda_devices()
    def query_once():
        try:
            result = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.free,memory.total,utilization.gpu", "--format=csv,noheader,nounits"], check=True, capture_output=True, text=True)
        except Exception as exc:
            print(f"Could not query nvidia-smi ({exc}).")
            if fallback_allow_any_visible_gpu_if_nvidia_smi_fails:
                return [dict(logical_id=i, physical_id=i, memory_used_mb=0, memory_free_mb=min_free_memory_mb, memory_total_mb=0, util_percent=0) for i in range(n_torch)]
            return []
        rows = []
        for line in result.stdout.strip().splitlines():
            parts = [part.strip() for part in line.split(",")]
            if len(parts) != 5:
                continue
            physical_id, memory_used_mb, memory_free_mb, memory_total_mb, util_percent = map(int, parts)
            if visible_physical_ids is not None:
                if physical_id not in visible_physical_ids:
                    continue
                logical_id = visible_physical_ids.index(physical_id)
            else:
                logical_id = physical_id
            if logical_id >= n_torch:
                continue
            rows.append(dict(logical_id=int(logical_id), physical_id=int(physical_id), memory_used_mb=int(memory_used_mb), memory_free_mb=int(memory_free_mb), memory_total_mb=int(memory_total_mb), util_percent=int(util_percent)))
        return rows
    samples = []
    for sample_idx in range(int(util_samples)):
        rows = query_once()
        if rows:
            samples.append(rows)
        if sample_idx + 1 < int(util_samples):
            time.sleep(float(sample_interval_sec))
    if not samples:
        return []
    by_logical_id = {}
    for rows in samples:
        for row in rows:
            logical_id = int(row["logical_id"])
            rec = by_logical_id.setdefault(logical_id, dict(logical_id=logical_id, physical_id=int(row["physical_id"]), memory_used_mb=int(row["memory_used_mb"]), memory_free_mb=int(row["memory_free_mb"]), memory_total_mb=int(row["memory_total_mb"]), util_samples=[]))
            rec["physical_id"] = int(row["physical_id"])
            rec["memory_used_mb"] = int(row["memory_used_mb"])
            rec["memory_free_mb"] = int(row["memory_free_mb"])
            rec["memory_total_mb"] = int(row["memory_total_mb"])
            rec["util_samples"].append(int(row["util_percent"]))
    idle_logical_ids = []
    print("Visible GPU status from utilization samples:")
    for logical_id in sorted(by_logical_id):
        rec = by_logical_id[logical_id]
        util_max = max(rec["util_samples"]) if rec["util_samples"] else 0
        enough_memory = rec["memory_free_mb"] >= int(min_free_memory_mb)
        is_idle = util_max <= int(max_util_percent) and enough_memory
        tag = "free" if is_idle else ("low-util-but-memory-full" if util_max <= int(max_util_percent) else "busy")
        print(f" - logical GPU {logical_id} / physical GPU {rec['physical_id']}: util_samples={rec['util_samples']}, max_util={util_max}%, free_mem={rec['memory_free_mb']}/{rec['memory_total_mb']} MiB, used_mem={rec['memory_used_mb']} MiB -> {tag}")
        if is_idle:
            idle_logical_ids.append(logical_id)
    return idle_logical_ids[:max_devices]


def augment_responses(x):
    if not model.training:
        return x
    if RESPONSE_NOISE_STD > 0:
        x = x + torch.randn_like(x) * float(RESPONSE_NOISE_STD)
    if ELECTRODE_DROPOUT_PROB > 0:
        keep = (torch.rand(x.shape[0], 1, x.shape[2], 1, device=x.device) > float(ELECTRODE_DROPOUT_PROB)).float()
        x = x * keep
    if TIME_DROPOUT_PROB > 0:
        keep = (torch.rand(x.shape[0], 1, 1, x.shape[3], device=x.device) > float(TIME_DROPOUT_PROB)).float()
        x = x * keep
    return x


def task_loss(pred, y):
    return F.cross_entropy(pred, y, label_smoothing=float(CLASS_LABEL_SMOOTHING))


task1_gpu_ids = get_idle_cuda_devices(max_devices=2)
if len(task1_gpu_ids) < 2:
    raise RuntimeError(
        "Task 1 is configured to train on two GPUs with max sampled utilization <= 1% "
        "and at least 6500 MiB free memory, "
        f"but fewer than two free GPUs were found. Free logical GPU ids found: {task1_gpu_ids}. "
        "Wait for active/idle CUDA jobs to finish or lower the requested GPU count."
    )

device = torch.device(f"cuda:{task1_gpu_ids[0]}")
model = nn.DataParallel(model, device_ids=task1_gpu_ids)
model.to(device)
print(f"Running exact-label loop-topology final-data Task 1 model on GPUs {task1_gpu_ids} with DataParallel.")
checkpoint_model = model.module if isinstance(model, nn.DataParallel) else model

run_name = datetime.now().strftime(f"task1_N5_DIV40_macro_f1_{SPLIT_MODE}_%Y%m%d_%H%M%S")
task1_output_root = Path.cwd() / "task1_outputs"
if not (Path.cwd() / "Final_Task_1.ipynb").exists() and "final_task_dir" in globals():
    task1_output_root = Path(final_task_dir) / "task1_outputs"
task1_output_dir = task1_output_root / run_name
task1_output_dir.mkdir(parents=True, exist_ok=True)
task1_live_log_path = task1_output_dir / "live_training.log"
_task1_log_file = open(task1_live_log_path, "a", buffering=1)
_task1_original_stdout = sys.stdout
_task1_original_stderr = sys.stderr
sys.stdout = TeeStream(_task1_original_stdout, _task1_log_file)
sys.stderr = TeeStream(_task1_original_stderr, _task1_log_file)
print(f"Task 1 output directory: {task1_output_dir}", flush=True)
print(f"Live training log: {task1_live_log_path}", flush=True)

best_model_path = f"/home/{getpass.getuser()}/best_model_final_task_1_N5_DIV40_macro_f1.pth"
run_best_model_path = task1_output_dir / "best_model_this_run.pth"
all_time_best_val_f1 = 0.0
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict) and checkpoint.get("model_type") == "task1_final_N5_DIV40_macro_f1_v1":
        all_time_best_val_f1 = float(checkpoint.get("best_val_macro_f1", checkpoint.get("best_val_f1", checkpoint.get("best_val_acc", 0.0))))
        print(f"Previous all-time best exact-label Task 1 validation macro-F1: {100 * all_time_best_val_f1:.2f}%")
    else:
        print("Found incompatible old Task 1 checkpoint; this run starts a fresh exact-label best.")

history_rows = []
current_run_best_val_f1 = 0.0
current_run_best_val_acc = 0.0
current_run_best_val_weighted_f1 = 0.0
current_run_best_val_micro_f1 = 0.0
current_run_best_f1_epoch = None
current_run_best_val_loss = float("inf")
current_run_best_predictions = None
current_run_best_targets = None
current_run_best_frequencies = None
current_run_best_networks = None
current_run_best_network_local_ids = None
completed_epochs = 0
interrupted = False
early_stopped = False
stop_reason = None
epochs_without_improvement = 0
best_val_f1_for_patience = 0.0
pbar = tqdm(range(1, n_epochs + 1), leave=True, dynamic_ncols=True)

try:
    for epoch in pbar:
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        for x, y, z, network_id in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            network_id = network_id.to(device, non_blocking=True)
            freq = ((z[:, 0].to(device, non_blocking=True) - freq_mean) / freq_std).float()
            optimizer.zero_grad(set_to_none=True)
            pred = model(augment_responses(x), freq, network_id)
            loss = task_loss(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss += float(loss.item())
            train_correct += int((pred.argmax(dim=-1) == y).sum().item())
            train_total += int(y.numel())
        train_loss /= max(1, len(train_loader))
        train_acc = train_correct / max(1, train_total)

        model.eval()
        val_losses = []
        val_pred = []
        val_targets = []
        val_frequencies_epoch = []
        val_networks_epoch = []
        with torch.no_grad():
            for x, y, z, network_id in val_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                network_id = network_id.to(device, non_blocking=True)
                freq = ((z[:, 0].to(device, non_blocking=True) - freq_mean) / freq_std).float()
                pred = model(x, freq, network_id)
                loss = task_loss(pred, y)
                val_losses.append(float(loss.item()))
                val_pred.append(pred.argmax(dim=-1).detach().cpu().numpy())
                val_targets.append(y.detach().cpu().numpy())
                val_frequencies_epoch.append(z[:, 0].detach().cpu().numpy())
                val_networks_epoch.append(network_id.detach().cpu().numpy())
        val_predictions = np.concatenate(val_pred)
        val_targets_np = np.concatenate(val_targets)
        val_frequencies_np = np.concatenate(val_frequencies_epoch).astype(np.float32)
        val_network_local_np = np.concatenate(val_networks_epoch).astype(np.int64)
        val_network_labels_np = np.array([available_networks[i] for i in val_network_local_np], dtype=np.int64)
        val_metrics, val_f1_by_pattern = task1_classification_metrics(val_targets_np, val_predictions, n_classes)
        val_acc = float(val_metrics["accuracy"])
        val_macro_f1 = float(val_metrics["macro_f1"])
        val_weighted_f1 = float(val_metrics["weighted_f1"])
        val_micro_f1 = float(val_metrics["micro_f1"])
        val_loss = float(np.mean(val_losses))
        scheduler.step(val_macro_f1)
        completed_epochs = epoch

        is_best_f1 = val_macro_f1 > current_run_best_val_f1
        improved_for_patience = val_macro_f1 > best_val_f1_for_patience + float(EARLY_STOP_MIN_DELTA)
        if improved_for_patience:
            best_val_f1_for_patience = float(val_macro_f1)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if is_best_f1:
            current_run_best_val_f1 = val_macro_f1
            current_run_best_val_acc = val_acc
            current_run_best_val_weighted_f1 = val_weighted_f1
            current_run_best_val_micro_f1 = val_micro_f1
            current_run_best_f1_epoch = epoch
            current_run_best_val_loss = val_loss
            current_run_best_predictions = val_predictions.copy()
            current_run_best_targets = val_targets_np.copy()
            current_run_best_frequencies = val_frequencies_np.copy()
            current_run_best_networks = val_network_labels_np.copy()
            current_run_best_network_local_ids = val_network_local_np.copy()
            payload = dict(
                model_type="task1_final_N5_DIV40_macro_f1_v1",
                model_state_dict=checkpoint_model.state_dict(),
                best_val_macro_f1=current_run_best_val_f1,
                best_val_f1=current_run_best_val_f1,
                best_val_weighted_f1=current_run_best_val_weighted_f1,
                best_val_micro_f1=current_run_best_val_micro_f1,
                best_val_acc=current_run_best_val_acc,
                best_val_loss=current_run_best_val_loss,
                primary_metric="macro_f1",
                best_epoch=epoch,
                run_name=run_name,
                available_networks=available_networks,
                network_to_local=network_to_local,
                split_mode=SPLIT_MODE,
                holdout_network=HOLDOUT_NETWORK if SPLIT_MODE == "leave_network_out" else None,
                n_max_electrodes=n_max_electrodes,
                n_time=n_time,
                mean=mean,
                std=std,
                freq_mean=freq_mean,
                freq_std=freq_std,
                coords_by_network=coords_by_network,
                electrode_circular_by_network=electrode_circular_by_network,
                adjacency_by_network=adjacency_by_network,
                electrode_static_by_network=electrode_static_by_network,
                electrode_mask_by_network=electrode_mask_by_network,
                class_prototypes=class_prototypes,
                class_counts_by_network=class_counts_by_network,
                class_label_smoothing=CLASS_LABEL_SMOOTHING,
                uses_pattern_circle_loss=False,
            )
            torch.save(payload, run_best_model_path)
            if current_run_best_val_f1 > all_time_best_val_f1:
                torch.save(payload, best_model_path)
                all_time_best_val_f1 = current_run_best_val_f1

        row = dict(
            epoch=epoch,
            train_loss=float(train_loss),
            train_acc=float(train_acc),
            train_acc_percent=float(100 * train_acc),
            val_loss=float(val_loss),
            val_macro_f1=float(val_macro_f1),
            val_macro_f1_percent=float(100 * val_macro_f1),
            val_weighted_f1=float(val_weighted_f1),
            val_weighted_f1_percent=float(100 * val_weighted_f1),
            val_micro_f1=float(val_micro_f1),
            val_micro_f1_percent=float(100 * val_micro_f1),
            val_acc=float(val_acc),
            val_acc_percent=float(100 * val_acc),
            learning_rate=float(optimizer.param_groups[0]["lr"]),
            epochs_without_improvement=int(epochs_without_improvement),
            is_best_f1_this_run=bool(is_best_f1),
        )
        for network in np.unique(val_network_labels_np):
            mask = val_network_labels_np == network
            network_metrics, _ = task1_classification_metrics(val_targets_np[mask], val_predictions[mask], n_classes)
            row[f"val_macro_f1_network_{int(network)}"] = float(network_metrics["macro_f1"])
            row[f"val_acc_network_{int(network)}"] = float(network_metrics["accuracy"])
        history_rows.append(row)
        pbar.set_description(
            f"epoch {epoch} train_acc={100*train_acc:.1f}% val_f1={100*val_macro_f1:.1f}% "
            f"best_f1={100*current_run_best_val_f1:.1f}% stale={epochs_without_improvement}"
        )
        if epoch % 5 == 0 or is_best_f1:
            print(
                f"Epoch {epoch:03d}: train_acc={100*train_acc:.2f}% val_macro_f1={100*val_macro_f1:.2f}% "
                f"val_acc={100*val_acc:.2f}% val_loss={val_loss:.4f} best_f1={100*current_run_best_val_f1:.2f}% "
                f"stale={epochs_without_improvement}/{EARLY_STOP_PATIENCE}",
                flush=True,
            )
        if epochs_without_improvement >= int(EARLY_STOP_PATIENCE):
            early_stopped = True
            stop_reason = f"early_stop_no_val_macro_f1_improvement_{EARLY_STOP_PATIENCE}_epochs_min_delta_{EARLY_STOP_MIN_DELTA}"
            print(f"Early stopping at epoch {epoch}: {stop_reason}", flush=True)
            break
except KeyboardInterrupt:
    interrupted = True
    print("Training interrupted; writing partial artifacts.", flush=True)
finally:
    summary = dict(
        status="interrupted" if interrupted else ("early_stopped" if early_stopped else "completed"),
        stop_reason=stop_reason,
        model_type="task1_final_N5_DIV40_macro_f1_v1",
        completed_epochs=int(completed_epochs),
        primary_metric="macro_f1",
        current_run_best_val_macro_f1=float(current_run_best_val_f1),
        current_run_best_val_f1=float(current_run_best_val_f1),
        current_run_best_val_weighted_f1=float(current_run_best_val_weighted_f1),
        current_run_best_val_micro_f1=float(current_run_best_val_micro_f1),
        current_run_best_val_acc=float(current_run_best_val_acc),
        current_run_best_f1_epoch=int(current_run_best_f1_epoch) if current_run_best_f1_epoch is not None else None,
        current_run_best_val_loss=float(current_run_best_val_loss),
        all_time_best_val_macro_f1=float(all_time_best_val_f1),
        available_networks=available_networks,
        split_mode=SPLIT_MODE,
        holdout_network=HOLDOUT_NETWORK if SPLIT_MODE == "leave_network_out" else None,
        val_frac=float(VAL_FRAC),
        n_train=int(len(train_dataset)),
        n_val=int(len(val_dataset)),
        batch_size=int(batch_size),
        learning_rate=float(learning_rate),
        weight_decay=float(weight_decay),
        class_label_smoothing=float(CLASS_LABEL_SMOOTHING),
        uses_pattern_circle_loss=False,
        uses_pattern_angle_head=False,
        uses_electrode_loop_topology=True,
        early_stop_patience=int(EARLY_STOP_PATIENCE),
        early_stop_min_delta=float(EARLY_STOP_MIN_DELTA),
        scheduler_patience=int(SCHEDULER_PATIENCE),
        device=str(device),
        task1_gpu_ids=[int(x) for x in task1_gpu_ids],
        uses_data_parallel=isinstance(model, nn.DataParallel),
        run_name=run_name,
        output_dir=str(task1_output_dir),
        best_model_this_run=str(run_best_model_path),
        all_time_best_model_path=best_model_path,
    )
    write_task1_artifacts(
        task1_output_dir,
        history_rows,
        summary,
        best_predictions=current_run_best_predictions,
        best_targets=current_run_best_targets,
        best_frequencies=current_run_best_frequencies,
        best_networks=current_run_best_networks,
        best_network_local_ids=current_run_best_network_local_ids,
    )
    print("Task 1 run summary:", flush=True)
    print(json.dumps(summary, indent=2), flush=True)
    if "_task1_log_file" in globals():
        sys.stdout = _task1_original_stdout
        sys.stderr = _task1_original_stderr
        _task1_log_file.close()
        print(f"Closed live training log: {task1_live_log_path}")




In [ ]:
# 4. Benchmarking

In [ ]:
# Compact post-training report for the most recent exact-label final-data run.
if "task1_output_dir" not in globals():
    raise RuntimeError("Run the training cell first.")
summary = json.loads((Path(task1_output_dir) / "summary.json").read_text())
print(json.dumps(summary, indent=2))
for name in [
    "train_class_counts_by_network.csv",
    "f1_summary.csv",
    "f1_by_pattern.csv",
    "accuracy_by_network.csv",
    "accuracy_by_pattern.csv",
    "accuracy_by_frequency.csv",
    "accuracy_by_network_pattern.csv",
]:
    path = Path(task1_output_dir) / name
    if path.exists():
        df = pd.read_csv(path)
        print("\n", name)
        sort_col = "f1" if "f1" in df.columns else ("accuracy" if "accuracy" in df.columns else df.columns[0])
        display(df.sort_values(sort_col).head(20))



In [ ]:
# Leave-network-out evaluation instructions.
# To test transfer to a completely unseen network, set in the setup cell:
#   SPLIT_MODE = "leave_network_out"
#   HOLDOUT_NETWORK = 6  # or 7/8/5, as available
# Then rerun cells from data preparation onward.
print("Leave-network-out is implemented via SPLIT_MODE='leave_network_out' and HOLDOUT_NETWORK=<network>.")
print("Available networks in the current load:", available_networks if 'available_networks' in globals() else 'run loading cell first')